In [3]:
#load the dataset
from pathlib import Path 

train_path = Path("../data/raw/train.dat")
test_path = Path("../data/raw/test.dat")

with open(train_path, "r", encoding="utf-8") as f:
    train_lines = f.readlines()

with open(test_path, "r", encoding="utf-8") as f:
    test_lines = f.readlines()

print("Training records:", len(train_lines))
print("Teatiing records:", len(test_lines))

print("\nSample training record:")
print(train_lines[0][:500])

Training records: 14438
Teatiing records: 14442

Sample training record:
4	Catheterization laboratory events and hospital outcome with direct angioplasty for acute myocardial infarction To assess the safety of direct infarct angioplasty without antecedent thrombolytic therapy, catheterization laboratory and hospital events were assessed in consecutively treated patients with infarctions involving the left anterior descending (n = 100 patients), right (n = 100), and circumflex (n = 50) coronary arteries. The groups of patients were similar for age (left anterior desce


In [4]:
#Extract the medical text
def extract_text(line):
    parts = line.strip().split(maxsplit=1)

    if len(parts) == 2:
        lebel, text = parts
        return text
    return ""

train_texts = [extract_text(line) for line in train_lines]
test_texts = [extract_text(line) for line in test_lines]

print(train_texts[0][:500])

Catheterization laboratory events and hospital outcome with direct angioplasty for acute myocardial infarction To assess the safety of direct infarct angioplasty without antecedent thrombolytic therapy, catheterization laboratory and hospital events were assessed in consecutively treated patients with infarctions involving the left anterior descending (n = 100 patients), right (n = 100), and circumflex (n = 50) coronary arteries. The groups of patients were similar for age (left anterior descend


In [5]:
#Basic cleaning
#lowercase, remove unnecassary whitespace
import re

def clean_text(text):
    text = text.lower()

    text = re.sub(r"\s+"," ", text)

    text = text.strip()

    return text

train_texts_clean = [clean_text(text) for text in train_texts]
test_texts_clean = [clean_text(text) for text in test_texts]

print(train_texts_clean[0][:500])

catheterization laboratory events and hospital outcome with direct angioplasty for acute myocardial infarction to assess the safety of direct infarct angioplasty without antecedent thrombolytic therapy, catheterization laboratory and hospital events were assessed in consecutively treated patients with infarctions involving the left anterior descending (n = 100 patients), right (n = 100), and circumflex (n = 50) coronary arteries. the groups of patients were similar for age (left anterior descend


In [6]:
#Inspect dataset statistics
train_words = sum(len(text.split()) for text in train_texts_clean)
test_words = sum(len(text.split()) for text in test_texts_clean)

print("Training documents:", len(train_texts_clean))
print("Testing documents:", len(test_texts_clean))

print("Training words:", train_words)
print("Testing words:", test_words)

Training documents: 14438
Testing documents: 14442
Training words: 2597932
Testing words: 2647098


In [8]:
#Save the cleaned corpus
processed_dir = Path("../data/processed")
processed_dir.mkdir(exist_ok=True)

with open(processed_dir / "train_clean.txt", "w", encoding="utf-8") as f:
    for text in train_texts_clean:
        f.write(text + "\n")

with open(processed_dir / "test_clean.txt","w",encoding="utf-8") as f:
    for text in test_texts_clean:
        f.write(text + "\n")


In [10]:
#Tokenization

from tensorflow.keras.preprocessing.text import Tokenizer
# Our medical dataset contains a very large number of unique words.
# Using every single word would make the final Dense layer very large
# and make training unnecessarily slow.
#
# So, for now, we keep the 20,000 most common words.
VOCAB_LIMIT = 20000

# oov_token = Out Of Vocabulary token.
#
# If the model later sees a word that was NOT present in the
# vocabulary, that word will be represented using <OOV>.
tokenizer = Tokenizer(
    num_words = VOCAB_LIMIT,
    oov_token="<OOV"
)
#WE only fit the tokenizer on the TRAINING data 
# bc the test set should represent unseen data

tokenizer.fit_on_texts(train_texts_clean)
#word_index is a dictionary containing
#word -> integer
word_index = tokenizer.word_index

vocab_size = min(VOCAB_LIMIT, len(word_index) + 1)

print("Total unique words found:", len(word_index))
print("Voculary size we will use:", vocab_size)

Total unique words found: 38626
Voculary size we will use: 20000


In [11]:
#Show the first 20 words learned by the tokenizer
for word, number in list(word_index.items())[:20]:
    print(word,"->",number)

<OOV -> 1
the -> 2
of -> 3
and -> 4
in -> 5
with -> 6
a -> 7
to -> 8
patients -> 9
was -> 10
were -> 11
for -> 12
or -> 13
is -> 14
by -> 15
that -> 16
than -> 17
0 -> 18
1 -> 19
from -> 20


In [12]:
#Convert every training document from words to numbers based on the numbers assigned already
train_sequences = tokenizer.texts_to_sequences(train_texts_clean)
test_sequences = tokenizer.texts_to_sequences(train_texts_clean)

print("original text: ")
print(train_texts_clean[0][:300])

print("\nSame text represented as numbers:")
print(train_sequences[0][:50])

original text: 
catheterization laboratory events and hospital outcome with direct angioplasty for acute myocardial infarction to assess the safety of direct infarct angioplasty without antecedent thrombolytic therapy, catheterization laboratory and hospital events were assessed in consecutively treated patients wi

Same text represented as numbers:
[1249, 901, 577, 4, 303, 271, 6, 797, 372, 12, 80, 146, 224, 8, 708, 2, 1365, 3, 797, 1095, 372, 118, 6707, 1792, 68, 1249, 901, 4, 303, 577, 11, 548, 5, 6344, 96, 9, 6, 3077, 1176, 2, 110, 489, 1416, 116, 292, 9, 234, 116, 292, 4]


In [13]:
#get the words back from the number
index_to_word={
    index: word
    for word, index in tokenizer.word_index.items()
}

sample_sequence = train_sequences[0][:15]

print("Numbers:")
print(sample_sequence)

print("\nConverted back to words: ")
for number in sample_sequence:
    print(index_to_word.get(number,"<OOV>"),end=" ")

Numbers:
[1249, 901, 577, 4, 303, 271, 6, 797, 372, 12, 80, 146, 224, 8, 708]

Converted back to words: 
catheterization laboratory events and hospital outcome with direct angioplasty for acute myocardial infarction to assess 

In [14]:
#context length basically means how many words the model is allowed to see before predicting the next word
CONTEXT_LENGTH = 20


In [27]:
#creating input X and output Y
import numpy as np
def create_training_sequences(sequences, context_length):
    X=[]
    Y=[]
    #sliding window
    for sequence in sequences:
        if len(sequence)<= context_length:
            continue
        for i in range(len(sequence)-context_length):
            input_words = sequence[i:i+context_length]
            next_word = sequence[i+context_length]

            X.append(input_words)
            Y.append(next_word)

    X = np.array(X,dtype=np.int32)
    Y = np.array(Y, dtype=np.int32)

    return X,Y
        


In [28]:
#testing sequence generation on a small sample
sample_train_sequences = train_sequences[:100]
X_sample,Y_sample = create_training_sequences(
    sample_train_sequences,
    CONTEXT_LENGTH
)
print("X shape: ",X_sample.shape)
print("Y shape:", Y_sample.shape)

X shape:  (17415, 20)
Y shape: (17415,)


In [ ]:

# VIEW ONE NEXT-WORD TRAINING EXAMPLE


example_number = 0


# Get one input sequence.
input_numbers = X_sample[example_number]

# Get its correct next word.
target_number = Y_sample[example_number]


print("INPUT NUMBERS:")
print(input_numbers)


print("\nINPUT WORDS:")

for number in input_numbers:
    print(index_to_word.get(number, "<OOV>"), end=" ")


print("\n\nCORRECT NEXT WORD:")

print(index_to_word.get(target_number, "<OOV>"))

INPUT NUMBERS:
[1249  901  577    4  303  271    6  797  372   12   80  146  224    8
  708    2 1365    3  797 1095]

INPUT WORDS:
catheterization laboratory events and hospital outcome with direct angioplasty for acute myocardial infarction to assess the safety of direct infarct 

CORRECT NEXT WORD:
angioplasty


In [31]:
#spliting training data into train and validation

from sklearn.model_selection import train_test_split

#we will split it as 90:10, 10% to see how well the model will perform
train_docs, val_docs = train_test_split(
    train_texts_clean,
    test_size = 0.10,
    random_state=42
)

print("Training document:",len(train_docs))
print("Validation documents:",len(val_docs))
print("TEsting documents:", len(test_texts_clean))

Training document: 12994
Validation documents: 1444
TEsting documents: 14442


In [32]:
#Creating tokenizer again but only uising training set and not validation set
from tensorflow.keras.preprocessing.text import Tokenizer
tokeninzer = Tokenizer(
    num_words=VOCAB_LIMIT,
    oov_token="<OOV>"
)
tokenizer.fit_on_texts(train_docs)

word_index = tokenizer.word_index

vocab_size = min(VOCAB_LIMIT, len(word_index)+1)

print("Total unique words found:", len(word_index))
print("Vocabulary size being used:", vocab_size)

#mapping the integer id back into words
index_to_word ={
    index: word
    for index, index in tokenizer.word_index.items()
}


Total unique words found: 38626
Vocabulary size being used: 20000


In [34]:
#convert train/validation/test text into numbers

train_sequences = tokenizer.texts_to_sequences(train_docs)
val_sequences = tokenizer.texts_to_sequences(val_docs)
test_sequences = tokenizer.texts_to_sequences(test_texts_clean)

print("Training documents converted:", len(train_sequences))
print("Validation documents converted:",len(val_sequences))
print("Testinf documents converted:", len(test_sequences))


Training documents converted: 12994
Validation documents converted: 1444
Testinf documents converted: 14442


In [35]:
#Generating the real training sequence
X_train, Y_train = create_training_sequences(
    train_sequences,
    CONTEXT_LENGTH
)
#Validation data
X_val,Y_val = create_training_sequences(
    val_sequences,
    CONTEXT_LENGTH
)
print("Training X shape:", X_train.shape)
print("Training Y shape:",Y_train.shape)

print()

print("Validation X shape:", X_val.shape)
print("Validation Y shape:", Y_val.shape)



Training X shape: (2153888, 20)
Training Y shape: (2153888,)

Validation X shape: (237549, 20)
Validation Y shape: (237549,)


In [37]:
#save tokenizer
import pickle
from pathlib import Path

#folder in which the preprocesseing related objects will be stored
processed_dir = Path("../data/processed")

with open(processed_dir/ "tokennizer.pkl","wb") as file:
    pickle.dump(tokenizer, file)

print("Tokenizer saved succesfully")

Tokenizer saved succesfully


In [38]:
#save preprocessing configuration
import json
config = {
    "vocab_limit":VOCAB_LIMIT,
    "vocab_size":vocab_size,
    "context_length": CONTEXT_LENGTH
}

with open(processed_dir/"config.json","w") as file:
    json.dump(config,file,indent=4)

print(config)

{'vocab_limit': 20000, 'vocab_size': 20000, 'context_length': 20}


In [40]:
import numpy as np
from pathlib import Path

#Folder where processed data is stored
processed_dir = Path("../data/processed")
#Save the input sequences 
# X_train contains the 20 previous words
np.save(
    processed_dir / "X_train.npy",
    X_train
)

#Save the correct next word for every training example
np.save(
    processed_dir / "Y_train.npy",
    Y_train
)

#Save validation input sequences
np.save(
    processed_dir / "X_val.npy",
    X_val
)

#Save validation targets
np.save(
    processed_dir / "Y_val.npy",
    Y_val
)
print("Training and validation arrays saved successfully")

Training and validation arrays saved successfully
